# ThermalTwin — One-Step Model Comparison

This notebook is a worked example of how to use the ThermalTwin toolbox end to end: load Netatmo data, engineer features, train several one-step-ahead indoor temperature predictors, and compare them.

It covers the `*_one_step` model family (`LinearOneStepPredictor`, `RidgeOneStepPredictor`, `NeuralOneStepPredictor`), which all consume the same flat, lag-based feature matrix produced by `FeatureEngineer`. See [`06_weather_only_comparison.ipynb`](06_weather_only_comparison.ipynb) for the windowed `*_weather_only` family.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from utils.data_loader import NetatmoDataLoader
from utils.feature_engineer import FeatureEngineer
from utils.split import DatasetSplitter
from utils.metrics import compute_metrics

from models.linear_one_step import LinearOneStepPredictor
from models.ridge_one_step import RidgeOneStepPredictor
from models.neural_one_step import NeuralOneStepPredictor

## 1. Load Data

In [ ]:
DATA_DIR = Path("../data/csv")

loader = NetatmoDataLoader(DATA_DIR)

loader.station_names()

In [ ]:
STATION = "Huis_19_06_2026_all"

df = loader.load_station(STATION)

df.head()

## 2. Feature Engineering

`FeatureEngineer` adds cyclical time-of-day/time-of-year features and lagged indoor/outdoor temperature columns, then builds the one-step-ahead target.

In [ ]:
engineer = FeatureEngineer(
    indoor_lags=48,
    outdoor_lags=48,
)

engineer.summary()

In [ ]:
df_features = engineer.transform(df)

In [ ]:
X, y = engineer.create_dataset(df_features)

print(X.shape)
print(y.shape)

## 3. Train / Validation / Test Split

`DatasetSplitter` splits chronologically (no shuffling), which is required for time series data.

In [ ]:
splitter = DatasetSplitter()

(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
) = splitter.split(X, y)

In [ ]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

## 4. Train and Compare One-Step Models

All predictors share the same `BasePredictor` interface (`fit(X_train, y_train, X_val, y_val, **kwargs)`), so they can be trained in a single loop — the sklearn-based models simply ignore the neural-network-only kwargs (`epochs`, `verbose`).

In [ ]:
models = {
    "Linear": LinearOneStepPredictor(),
    "Ridge": RidgeOneStepPredictor(alpha=1.0),
    "Neural": NeuralOneStepPredictor(n_features=X_train.shape[1]),
}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train, X_val, y_val, epochs=30, verbose=0)

## 5. Evaluate

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    val_mae, val_rmse = compute_metrics(y_val, y_val_pred)
    test_mae, test_rmse = compute_metrics(y_test, y_test_pred)

    results.append({
        "model": name,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
    })

    predictions[name] = y_test_pred

results_df = pd.DataFrame(results).set_index("model")

In [ ]:
results_df

## 6. Visualize Results

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(y_test.values, label="Actual", color="black", linewidth=2)

for name, pred in predictions.items():
    plt.plot(pred, label=name, alpha=0.8)

plt.legend()
plt.title("One-step-ahead indoor temperature prediction — model comparison")
plt.xlabel("Time step")
plt.ylabel("Indoor temperature (°C)")
plt.show()

In [ ]:
results_df[["test_mae", "test_rmse"]].plot(kind="bar", figsize=(8, 5))

plt.title("Test set error by model")
plt.ylabel("°C")
plt.xticks(rotation=0)
plt.show()

As also noted in the project [README](../README.md), the previous indoor temperature tends to be an extremely strong one-step-ahead predictor on its own, given the building's thermal inertia — keep that in mind when comparing these models against how much they actually improve on that.